# Alignment files
In the previous workshop we discussed a common sequence file formats, FASTA/Q, where each entry in the file represents a signle biological sequence (which could be a chromosome, a sequencing read, etc.). Such files are often a "first step" or the starting point in a bioinformatic genomic workflow. In this workshop, we will continue on in our workflow to discuss some common types of alignment files, which can be generated in a variety of ways and like sequence files can represent a variety of things: whole genome alignments to identify rearrangements, RNAseq reads vs reference transcriptome to calculate differential expression, genomic resequencing mapped to a genome to identify polymorphism, even certain sequencers will output reads in an alignment format. 

Commonly, alignment files have a "reference" (sometimes also called a "target") sequence, with "query" sequence(s) mapped against the reference. In this case, mapping coordinates (i.e. where the align to) are relative to the reference sequence. We can also have alignments without a reference sequence, in which case sequences in the alignment are "equal" and coordinates are specific to each sequence.

## Aligned FASTA files

## Intro to SAM/BAM format
SAM (Sequence Alignment/Map) format is one of the most common file formats produced by many different pieces of alignment software, both for long and short read sequence data. A number of different programs can output alignments in this format (e.g. [BWA](https://github.com/lh3/bwa), [STAR](https://github.com/alexdobin/STAR), [minimap2](https://github.com/lh3/minimap2)) and which you choose will vary based on your data type and experimental design, but the alignment file created will likely be interchangeable. In all cases, one sequence is designated as the reference, with queries aligned against it.

It is a tab delineated text file, with 11 mandatory fields, or columns, (listed below), with a 12th column containing optional "tags" with supplemental or aligner-specific information. SAM files are human readable, but can be quite large. An alternate format is the Binary Alignment/Map (BAM) file, which is binary compressed and not human readable, but is more compact and efficient to work with. Most pipelines will use BAM format over SAM, and for sotring alignments long-term BAM is usually preferable as it uses up less storage space. Converting between BAM and SAM is easy, and we will show how to down so further on.

| **Column** | **Description**                        |
|------------|----------------------------------------|
| 1          | Read name                              |
| 2          | Bitwise flag                           |
| 3          | Reference name                         |
| 4          | Leftmost mapping position              |
| 5          | MAPQ quality score                     |
| 6          | CIGAR string                           |
| 7          | Name of 2nd read in pair               |
| 8          | Position of 2nd read in pair           |
| 9          | Length of mapping segment              |
| 10         | Sequence of segment                    |
| 11         | Phred33 quality score at each position |
| 12         | Optional tags                          |

In addition to these tab-separated fields, BAM/SAM files also have header lines at their starts, which are denoted by a `@` character and a two letter code. Header lines contain metadata about the alignment, such as whether alignments are sorted or grouped, reference sequence information, and definitions for any tags that are used in the 12th column. While not strictly required, many downstream analysis tools require BAM/SAM files to have a header, and will thrown an error if not present. A full description of BAM/SAM format can be found [here](https://samtools.github.io/hts-specs/SAMv1.pdf).

Let's take a look at what these files look like.

> **Exercise** 
> Use `less` to open SAM file (use `-S` to wrap lines). Do the same for BAM file.

In [ ]:
# Command here

In [ ]:
#@title Solution {display-mode: "form"}
less -S ph.sam
less -S ph.bam

Just like any file, we can use our normal command line tools to visualize SAM files...however, it is a little clunky. Also, notice that when we try to open the BAM file, we get a scrambled mess; this is because as mentioned, BAM is a binary compressed format and is not human readable (which the shell will warn you of when trying to open the file). Thus, when working with BAM/SAM alignments, it is much easier to use a toolkit specilized for these types of files!

## SAMtools
[SAMtools](http://www.htslib.org/doc/samtools.html) is a suite of programs that are extremely useful for processing mapped reads and for downstream analysis. As stated above, BAM/SAM files from different programs are (mostly) interchangeable, so `samtools` will work with a file BAM/SAM file no matter what program produced it. Note that for this workshop we have already installed `samtools`, but to run it elsewhere you will need to install it yourself. It has a ton of functions (which you can check out on the [manual page](http://www.htslib.org/doc/samtools.html)), but we will go through several of the most common uses.

### samtools view
As the name suggests, this command lets you view the content of a SAM **or** BAM file. Let's take a look at a file by opening it with `view` and piping it to `head` to display just the first five lines.

In [ ]:
samtools view ph.bam | head -n 5

Note that even tho we are opening a BAM file, `view` automatically converts it into readable SAM format! This is how we convert BAM --> SAM. To convert SAM --> BAM, we can use the `-b` argument to instead output in BAM format, along with the `-o` argument to output to a file instead of STDOUT.

```
samtools view -b -o ph.bam ph.sam
```

By default, something important is missing when we save output to a file this way...can you figure out what it is?

<details><summary>Solution</summary>

The output file lacks a header! As mentioned, headers are vital for many downstream tools. 

> **Exercise** 
> Check the samtools documentation and find the correct option to add a header
  
</details>

In [ ]:
# command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -h -b -o ph.bam ph.sam

#### SAM flags and filtering
The second column in a BAM/SAM file is the *bitwise flag*. The flag value is an integer, which is the sum of a series of decimal values that give information about how a read is mapped.

| **Integer** | **Description**                |
|-------------|--------------------------------|
| 1           | read is paired                 |
| 2           | read mapped in proper pair     |
| 4           | read unmapped                  |
| 8           | mate is unmapped               |
| 16          | read on reverse strand         |
| 32          | mate on reverse strand         |
| 64          | first read in pair             |
| 128         | second read in pair            |
| 256         | not primary alignment          |
| 512         | alignment fails quality checks |
| 1024        | PCR or optical duplicate       |
| 2048        | supplementary alignment        |

So e.g., for a paired-end mapping data set, a flag = **99** (1+2+32+64) means the read is mapped along with its mate (1 and 2) and in the proper orientation (32 and 64). Don't worry about memorizing these, there are plenty of tools online that decode these flags for you, such as right [here](https://broadinstitute.github.io/picard/explain-flags.html).

While you don't need to know all the SAM flags, if there is one flag that is useful to have memorized it is **4**, which means the read is **unmapped**. Unmapped reads are most often filtered out, as many programs used in downstream analysis of SAM/BAM files only want mapped reads (and also to save space on disk!). You can filter reads containing a given flag using the `-f` (only take reads that match given flags) and `-F` (only take reads that do **NOT** match given flag) options in `samtools view`.

> Run the code block below to remove unmapped reads from the **SAM** file and display the first few reads retained:

In [ ]:
samtools view -F 4 data/file.sam | head

This removes any read that contains the 4 flag (e.g. 77, 141, etc.). You can filter on any other criteria using flags as well, e.g. only gets reads that map in proper pair:

```
samtools view -f 2 -h data/file.sam`
```

Note this uses `-f`, not `-F`, which **RETAINS** reads with those flags rather than filtering them!

> **Exercise**:
> In the code block below, count how many reads in the SAM file are mapped in their proper pairs vs not proper pairs?

In [ ]:
# command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -f 2 data/file.sam | wc -l
samtools view -F 2 data/file.sam | wc -l

### Sorting and indexing a BAM file
BAM files can get extremely large, over hundreds of GB in some cases, and by default are unsorted, which makes any task where we need to search for specific regions too computationally intensive. To overcome this, we can use two other functions of `samtools`, `sort` and `index`. This will create a BAM index `.bai` file, which allows quick lookup even for very large BAM files.

In [ ]:
# -o: This option tells samtools sort to print the output to the provided file rather than to the screen
samtools sort -o file.sorted.bam file.bam

#samtools index does not have an -o option, and will automatically create an index file with the same name as the input BAM file with a .bai extension
samtools index file.sorted.bam

This code will create a new `.sorted.bam` file that we create the index for, as `samtools index` requires a coordinate-sorted file. Any downstream program that refers to specific regions of interest in a BAM file, such as visualization tools like IGV (discussed later) or other `samtools` functions will require this index. For example, we can specify *specific region(s)* when using `samtools view` to only print alignments that overlap the specified region. Regions are listed at the end of the `view` command with the format `reference name:start position-end position` (if start and end are not specified, it reports all alignments to the reference sequence): 

In [ ]:
#Outputs all alignments that are mapped to chromosome 1 and saves them to a new BAM file
samtools view -o ph.chr1.bam ph.bam chr1

#Outputs all alignments that are mapped from bases 1000 to 2000 (inclusive) on chromosome 1 and saves them to a new BAM file
samtools view -o ph.chr1:1000-2000.bam ph.bam chr1:1000-2000

> PRACTICE: Putting it all together!

Let's take everything that we have learned and organize it into what a typical workflow might look like. Assume that we have already aligned the data and the aligner we used outputs the alignment file in SAM format. We want to go from our initial SAM file and end up with a *sorted, indexed BAM file with only the mapped reads retained*. Try inputting the commands yourself, then we will walk through it together.

> **Exercise**:
> 1. Convert the `data/file.sam` **SAM** file to **BAM** format while retaining the header and removing unmapped reads, then sort the file. Call the new file `file.mapped.sorted.bam`.
> Bonus: `samtools` commands can also make use of **pipes** (`|`) to avoid writing intermediate files!
> 2. Index the newly created sorted **BAM** file.

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}

## Convert to BAM with header and without unmapped reads and then sort
samtools view -h -b -o file.mapped.bam -F 4 data/file.sam
samtools sort -o file.mapped.sorted.bam file.mapped.bam

#Or, in a single line:
samtools view -h -b -F 4 data/file.sam | samtools sort -o file.mapped.sorted.bam 

## Index the new BAM file
samtools index file.mapped.sorted.bam

### More useful `samtools` utilities
We have our nice sorted and indexed BAM file, now what are some other useful pieces of information we can pull out of it? In many kinds of bioinformatic analysis, we will be interested in the **read coverage** and **read depth** in a particular region. "Coverage" and "depth" are often used interchangably but are technically are different concepts. "Coverage" is defined as the percentage of positions that have *at least one base aligned to it* (think of it as how much sequence is covered by mapped reads), while "depth" can be thought of as the redundancy of coverage (i.e. how many bases are aligned to a particular sequence). Aligning vs a reference and calculating coverage and/or depth for an alignment is the starting point for mainy kinds of analysis, such as:

- Identifying misassembled regions in a genome assembly
- Calculating differential expression of transcripts
- Calling single nucleotide polymorphisms or structural variants within a population 

To calculate this, we can use `samtools coverage` and `samtools depth`!

### Visualizing alignments with IGV

## What's wrong with my BAM file?